# CASMI26 EXP-0009 S1: z(analog) + 0.5 z(frag) + 文献量 prior

- 生成元: `scripts/casmi26_prepare_kernel.py` (git `0248dfb`)
- fold-safe に組み直した公開 baseline (haideptry V17) の Channel 1–3。FPNet と学習済みランカーは使わない
- 候補 pool: EXP-0008 の pool + PubChem 本体から窓ごとに文献量の上位 100 件
- ローカル CV: np Class 2 0.734 / np Class 1 0.797 / fold0 0.626

In [ ]:
MODE = 'analog_frag_prior'
POOL = 'exp0009'
EMBEDDED = {"casmi26/__init__.py": "", "casmi26/metrics/__init__.py": "from casmi26.metrics.breakdown import (\n    SubmissionRow,\n    canonical_inchikey14,\n    mrr_at_k,\n    mrr_at_k_breakdown,\n    score_submission,\n)\n\n__all__ = [\n    \"SubmissionRow\",\n    \"canonical_inchikey14\",\n    \"mrr_at_k\",\n    \"mrr_at_k_breakdown\",\n    \"score_submission\",\n]\n", "casmi26/metrics/breakdown.py": "\"\"\"MRR@25 と構成要素の計算 (CASMI26 公式スコアラの再現と因数分解)。\n\n評価指標:\n    MRR@25 = (1/U) * Σ_u 1/rank_u\n\n    U は test の unique molecule_id 数、rank_u は分子 u に対する ranked list で\n    最初に正解となった構造の順位 (1〜25)。25 候補中に正解がなければ 0 点。\n\n正解判定キー:\n    canonical_inchikey14 (RDKit でタウトマー正規化した SMILES から\n    MolToInchiKey → 先頭 14 文字)。\n\n    train の `inchikey14` 列は RDKit canonicalization 未適用のため、\n    正解判定にはこちらを使う (kernel 分析 `kitopl/casmi-2026-data-pitfalls` 参照)。\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Iterable, Sequence\n\nfrom rdkit import Chem, RDLogger\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\n\nRDLogger.DisableLog(\"rdApp.*\")\n\nK_MAX = 25\n\n_TAUTOMER = rdMolStandardize.TautomerEnumerator()\n\n\ndef canonical_inchikey14(smiles: str) -> str | None:\n    \"\"\"SMILES → canonical InChIKey14。パース失敗時は None。\"\"\"\n    if not smiles:\n        return None\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None:\n        return None\n    try:\n        mol = _TAUTOMER.Canonicalize(mol)\n    except Exception:\n        pass\n    inchi_key = Chem.MolToInchiKey(mol)\n    if not inchi_key:\n        return None\n    return inchi_key[:14]\n\n\n@dataclass(frozen=True)\nclass SubmissionRow:\n    \"\"\"提出行の 1 分子分 (molecule_id, 候補 SMILES 列、正解 SMILES)。\"\"\"\n\n    molecule_id: str\n    candidate_smiles: Sequence[str]\n    truth_smiles: str\n\n\ndef _first_hit_rank(\n    candidate_keys: Sequence[str | None], truth_key: str | None, k_max: int\n) -> int | None:\n    \"\"\"truth_key が最初に出た順位 (1-based)。なければ None。\"\"\"\n    if truth_key is None:\n        return None\n    for rank, key in enumerate(candidate_keys[:k_max], start=1):\n        if key is not None and key == truth_key:\n            return rank\n    return None\n\n\ndef mrr_at_k(rows: Iterable[SubmissionRow], *, k_max: int = K_MAX) -> float:\n    \"\"\"提出 rows から MRR@k を計算 (0 件の分子は 0 点)。\"\"\"\n    rows_list = list(rows)\n    if not rows_list:\n        return 0.0\n    total = 0.0\n    for row in rows_list:\n        truth_key = canonical_inchikey14(row.truth_smiles)\n        cand_keys = [canonical_inchikey14(s) for s in row.candidate_smiles]\n        rank = _first_hit_rank(cand_keys, truth_key, k_max)\n        if rank is not None:\n            total += 1.0 / rank\n    return total / len(rows_list)\n\n\n@dataclass(frozen=True)\nclass BreakdownResult:\n    \"\"\"MRR@k 因数分解結果。\n\n    rank_counts[r] は \"正解が rank r で見つかった分子数\"。rank_counts[0] は圏外。\n    sum / count = MRR@k。\n\n    class_contributions は optional で、クラスラベル (Class 1/2/3 相当) ごとの\n    MRR@k 寄与 (sum of 1/rank / U_total) を返す。U_total は全分子数 (母数固定)。\n    \"\"\"\n\n    k_max: int\n    count: int\n    mrr: float\n    top1_accuracy: float\n    rank_counts: dict[int, int]\n    class_contributions: dict[str, float] | None = None\n    class_counts: dict[str, int] | None = None\n    class_mrr: dict[str, float] | None = None\n\n\ndef mrr_at_k_breakdown(\n    rows: Sequence[SubmissionRow],\n    *,\n    k_max: int = K_MAX,\n    class_labels: Sequence[str] | None = None,\n) -> BreakdownResult:\n    \"\"\"提出 rows の MRR@k を因数分解。\n\n    class_labels が与えられれば rows と同長で各分子のクラスラベルを渡し、\n    クラスごとの寄与と MRR を返す。全体 MRR への寄与は \"クラスの sum(1/rank) / U_total\"\n    で、U_total は rows 全体の母数 (sum すると全体 MRR と一致)。\n    \"\"\"\n    if class_labels is not None and len(class_labels) != len(rows):\n        raise ValueError(\"class_labels length must match rows length\")\n\n    rank_counts: dict[int, int] = {0: 0}\n    for r in range(1, k_max + 1):\n        rank_counts[r] = 0\n\n    class_sum: dict[str, float] = {}\n    class_count: dict[str, int] = {}\n\n    total_sum = 0.0\n    top1 = 0\n    for i, row in enumerate(rows):\n        truth_key = canonical_inchikey14(row.truth_smiles)\n        cand_keys = [canonical_inchikey14(s) for s in row.candidate_smiles]\n        rank = _first_hit_rank(cand_keys, truth_key, k_max)\n        rr = 0.0 if rank is None else 1.0 / rank\n        total_sum += rr\n        if rank == 1:\n            top1 += 1\n        if rank is None:\n            rank_counts[0] += 1\n        else:\n            rank_counts[rank] += 1\n\n        if class_labels is not None:\n            label = class_labels[i]\n            class_sum[label] = class_sum.get(label, 0.0) + rr\n            class_count[label] = class_count.get(label, 0) + 1\n\n    n = len(rows)\n    mrr = total_sum / n if n else 0.0\n    top1_acc = top1 / n if n else 0.0\n\n    class_contributions: dict[str, float] | None = None\n    class_mrr: dict[str, float] | None = None\n    class_counts: dict[str, int] | None = None\n    if class_labels is not None and n:\n        class_contributions = {k: v / n for k, v in class_sum.items()}\n        class_mrr = {\n            k: (class_sum[k] / class_count[k]) if class_count[k] else 0.0 for k in class_sum\n        }\n        class_counts = dict(class_count)\n\n    return BreakdownResult(\n        k_max=k_max,\n        count=n,\n        mrr=mrr,\n        top1_accuracy=top1_acc,\n        rank_counts=rank_counts,\n        class_contributions=class_contributions,\n        class_counts=class_counts,\n        class_mrr=class_mrr,\n    )\n\n\ndef score_submission(\n    submission_smiles: dict[str, Sequence[str]],\n    truth_smiles: dict[str, str],\n    *,\n    k_max: int = K_MAX,\n) -> BreakdownResult:\n    \"\"\"submission dict と truth dict から MRR@k 因数分解結果を返す。\n\n    submission_smiles[molecule_id] は候補 SMILES 列 (`;` 連結ではなく list)。\n    truth_smiles[molecule_id] は正解 SMILES。\n    truth にない molecule_id は無視、truth にあって submission にない分子は 0 点扱い。\n    \"\"\"\n    rows: list[SubmissionRow] = []\n    for mol_id, truth in truth_smiles.items():\n        cands = submission_smiles.get(mol_id, [])\n        rows.append(\n            SubmissionRow(\n                molecule_id=mol_id,\n                candidate_smiles=list(cands),\n                truth_smiles=truth,\n            )\n        )\n    return mrr_at_k_breakdown(rows, k_max=k_max)\n", "casmi26/baseline/__init__.py": "\"\"\"fold-safe に組み直した公開 baseline (haideptry V17) の部品。\"\"\"\n", "casmi26/baseline/chem.py": "\"\"\"adduct → neutral mass 変換と構造指紋 (baseline V17 Cell 4 / Cell 6 の移植)。\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom rdkit import Chem, DataStructs, RDLogger\nfrom rdkit.Chem import rdFingerprintGenerator\n\nRDLogger.DisableLog(\"rdApp.*\")  # type: ignore[attr-defined]\n\nMASS = dict(\n    C=12.0,\n    H=1.00782503207,\n    N=14.0030740048,\n    O=15.9949146196,\n    P=30.97376163,\n    S=31.97207100,\n    F=18.99840322,\n    Cl=34.96885268,\n    Br=78.9183371,\n    I=126.904473,\n    Na=22.9897692809,\n    K=38.96370668,\n    Si=27.9769265325,\n    B=11.0093054,\n    Se=79.9165213,\n)\nE = 0.00054857990\nPROTON = MASS[\"H\"] - E\nH2O = 2 * MASS[\"H\"] + MASS[\"O\"]\nNH4 = MASS[\"N\"] + 4 * MASS[\"H\"]\nFORMATE = MASS[\"C\"] + 2 * MASS[\"H\"] + 2 * MASS[\"O\"]\nACETATE = 2 * MASS[\"C\"] + 4 * MASS[\"H\"] + 2 * MASS[\"O\"]\n\n# adduct: (n_M, charge, 付加質量)\nADDUCTS: dict[str, tuple[int, int, float]] = {\n    \"[M+H]+\": (1, 1, PROTON),\n    \"[M+NH4]+\": (1, 1, NH4 - E),\n    \"[M+Na]+\": (1, 1, MASS[\"Na\"] - E),\n    \"[M+K]+\": (1, 1, MASS[\"K\"] - E),\n    \"[M-H2O+H]+\": (1, 1, PROTON - H2O),\n    \"[M-2H2O+H]+\": (1, 1, PROTON - 2 * H2O),\n    \"[M+2H]2+\": (1, 2, 2 * PROTON),\n    \"[M]+\": (1, 1, -E),\n    \"[M-H2O]+\": (1, 1, -E - H2O),\n    \"[M+CH3OH+H]+\": (1, 1, PROTON + MASS[\"C\"] + 4 * MASS[\"H\"] + MASS[\"O\"]),\n    \"[M+CH3CN+H]+\": (1, 1, PROTON + 2 * MASS[\"C\"] + 3 * MASS[\"H\"] + MASS[\"N\"]),\n    \"[M-H]-\": (1, 1, -PROTON),\n    \"[M-H2O-H]-\": (1, 1, -PROTON - H2O),\n    \"[M+CH2O2-H]-\": (1, 1, FORMATE - PROTON),\n    \"[M+C2H4O2-H]-\": (1, 1, ACETATE - PROTON),\n    \"[M+Cl]-\": (1, 1, MASS[\"Cl\"] + E),\n    \"[M]-\": (1, 1, E),\n    \"[M-2H]-\": (1, 2, -2 * PROTON),\n    \"[M+Na-2H]-\": (1, 1, MASS[\"Na\"] - 2 * PROTON),\n    \"[2M+H]+\": (2, 1, PROTON),\n    \"[2M+Na]+\": (2, 1, MASS[\"Na\"] - E),\n    \"[2M+NH4]+\": (2, 1, NH4 - E),\n    \"[2M+K]+\": (2, 1, MASS[\"K\"] - E),\n    \"[2M-H]-\": (2, 1, -PROTON),\n    \"[2M+CH2O2-H]-\": (2, 1, FORMATE - PROTON),\n    \"[2M+C2H4O2-H]-\": (2, 1, ACETATE - PROTON),\n    \"[2M+Na-2H]-\": (2, 1, MASS[\"Na\"] - 2 * PROTON),\n    \"[3M+H]+\": (3, 1, PROTON),\n    \"[3M-H]-\": (3, 1, -PROTON),\n}\n\n\ndef neutral_mass(mz: np.ndarray, adduct: np.ndarray) -> np.ndarray:\n    \"\"\"precursor m/z と adduct から中性分子の monoisotopic mass。未知 adduct は nan。\"\"\"\n    out = np.full(len(mz), np.nan)\n    ad = np.asarray(adduct, dtype=object)\n    for a, (n, z, d) in ADDUCTS.items():\n        m = ad == a\n        if m.any():\n            out[m] = (mz[m] * z - d) / n\n    return out\n\n\ndef corrected_mz(\n    mz: np.ndarray, modes: np.ndarray, offset_pos: float, offset_neg: float\n) -> np.ndarray:\n    \"\"\"ionization_mode ごとの系統ずれ (Da) を precursor m/z から引く。\"\"\"\n    pos = np.asarray(modes, dtype=object) == \"positive\"\n    return np.asarray(mz, np.float64) - np.where(pos, offset_pos, offset_neg)\n\n\n_MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\n\n\ndef morgan_fp(smiles: str) -> DataStructs.ExplicitBitVect | None:\n    \"\"\"Morgan r2 2048-bit。baseline は 6,930-bit の複合指紋だが、ここでは Tanimoto 用途に絞る。\"\"\"\n    mol = Chem.MolFromSmiles(smiles) if smiles else None\n    if mol is None:\n        return None\n    return _MORGAN.GetFingerprint(mol)\n", "casmi26/baseline/fragments.py": "\"\"\"MetFrag-lite: 1〜2 本の結合切断で生じる断片質量がピークをどれだけ説明するか。\n\nbaseline V17 Cell 6 の移植 (Kaggle 公開 notebook, Apache 2.0)。\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom rdkit import Chem, RDLogger\n\nfrom casmi26.baseline.chem import MASS, PROTON\n\nRDLogger.DisableLog(\"rdApp.*\")  # type: ignore[attr-defined]\n\nH_ATOM = MASS[\"H\"]\n\n\ndef _mol_graph(smi: str) -> tuple[np.ndarray, list[tuple[int, int]], int] | None:\n    m = Chem.MolFromSmiles(smi)\n    if m is None:\n        return None\n    n = m.GetNumAtoms()\n    w = np.zeros(n)\n    for a in m.GetAtoms():\n        w[a.GetIdx()] = MASS.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H_ATOM\n    if (w == 0).any():\n        return None\n    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]\n    return w, bonds, n\n\n\ndef _components(n: int, bonds: list[tuple[int, int]], drop: set[int]) -> list[list[int]]:\n    adj: list[list[int]] = [[] for _ in range(n)]\n    for i, (a, b) in enumerate(bonds):\n        if i in drop:\n            continue\n        adj[a].append(b)\n        adj[b].append(a)\n    seen = np.zeros(n, bool)\n    comps = []\n    for s in range(n):\n        if seen[s]:\n            continue\n        stack = [s]\n        seen[s] = True\n        cur = [s]\n        while stack:\n            u = stack.pop()\n            for v in adj[u]:\n                if not seen[v]:\n                    seen[v] = True\n                    stack.append(v)\n                    cur.append(v)\n        comps.append(cur)\n    return comps\n\n\ndef fragment_masses(smi: str, max_breaks: int = 2, max_bonds: int = 34) -> np.ndarray:\n    try:\n        g = _mol_graph(smi)\n    except Exception:\n        return np.zeros(0)\n    if g is None:\n        return np.zeros(0)\n    w, bonds, n = g\n    nb = len(bonds)\n    if nb == 0 or nb > max_bonds:\n        return np.array([w.sum()])\n    out = {float(w.sum())}\n    for i in range(nb):\n        for c in _components(n, bonds, {i}):\n            out.add(float(w[c].sum()))\n    if max_breaks >= 2:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for c in _components(n, bonds, {i, j}):\n                    out.add(float(w[c].sum()))\n    if max_breaks >= 3:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for k in range(j + 1, nb):\n                    for c in _components(n, bonds, {i, j, k}):\n                        out.add(float(w[c].sum()))\n    return np.array(sorted(out))\n\n\ndef explain_score(\n    frag_mass: np.ndarray,\n    peak_mz: np.ndarray,\n    peak_int: np.ndarray,\n    mode: float = 1.0,\n    tol: float = 0.01,\n    h_shifts: tuple[int, ...] = (-2, -1, 0, 1, 2),\n) -> float:\n    \"\"\"sqrt(強度) 重みで、断片 ion に ±tol で一致したピークの割合。\"\"\"\n    if len(frag_mass) == 0 or len(peak_mz) == 0:\n        return 0.0\n    charge = PROTON if mode > 0 else -PROTON\n    ion = np.sort(np.concatenate([frag_mass + dh * H_ATOM + charge for dh in h_shifts]))\n    w = np.sqrt(np.asarray(peak_int, float))\n    tot = w.sum()\n    if tot <= 0:\n        return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)\n", "casmi26/baseline/spectra.py": "\"\"\"スペクトル前処理と entropy similarity (numba)。\n\n`haideptry/enveda-casmi-2026-fast-spectral-cosine-baseline` (V17) の Cell 3 を移植。\n差分: ライブラリ側スペクトルは検索のたびに clean せず、`clean_library` で 1 回だけ\n前処理して保持する (検索結果は同一)。\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pyarrow.parquet as pq\nfrom numba import njit, prange\n\nfrom casmi26.baseline.chem import neutral_mass\n\n\n@dataclass(frozen=True)\nclass CleanConfig:\n    floor: float = 0.002  # base peak 比 0.2% 未満を捨てる\n    topk: int = 256\n    power: float = 1.0\n    ent_weight: bool = True\n    mz_tol: float = 0.01\n    # > 0 なら m/z > precursor_mz - precursor_window のピーク (precursor 本体・同位体・\n    # precursor より重いピーク) を類似度計算の前に除く。0 は V17 と同じ (除かない)\n    precursor_window: float = 0.0\n\n\n@njit(cache=True, fastmath=True)\ndef clean(mz, it, floor, topk, power, ent_weight):\n    n = len(mz)\n    if n == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32)\n    mx = 0.0\n    for i in range(n):\n        if it[i] > mx:\n            mx = it[i]\n    if mx <= 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32)\n    thr = floor * mx\n    c = 0\n    for i in range(n):\n        if it[i] >= thr:\n            c += 1\n    idx = np.empty(c, np.int64)\n    j = 0\n    for i in range(n):\n        if it[i] >= thr:\n            idx[j] = i\n            j += 1\n    if c > topk:\n        v = np.empty(c, np.float32)\n        for i in range(c):\n            v[i] = it[idx[i]]\n        o = np.argsort(v)[c - topk :]\n        k2 = np.empty(topk, np.int64)\n        for i in range(topk):\n            k2[i] = idx[o[i]]\n        k2.sort()\n        idx = k2\n        c = topk\n    om = np.empty(c, np.float32)\n    oi = np.empty(c, np.float32)\n    s = 0.0\n    for i in range(c):\n        om[i] = mz[idx[i]]\n        v2 = it[idx[i]] ** power\n        oi[i] = v2\n        s += v2\n    if s > 0:\n        for i in range(c):\n            oi[i] /= s\n    if ent_weight:\n        S = 0.0\n        for i in range(c):\n            if oi[i] > 0:\n                S -= oi[i] * np.log(oi[i])\n        if S < 3.0:\n            w = 0.25 + 0.25 * S\n            s2 = 0.0\n            for i in range(c):\n                oi[i] = oi[i] ** w\n                s2 += oi[i]\n            if s2 > 0:\n                for i in range(c):\n                    oi[i] /= s2\n    return om, oi\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim(qmz, qp, cmz, cp, tol):\n    i = 0\n    j = 0\n    n = len(qmz)\n    m = len(cmz)\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0:\n            SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0:\n            SB -= cp[x] * np.log(cp[x])\n    SAB = 0.0\n    tot = 0.0\n    buf = np.empty(n + m, np.float64)\n    b = 0\n    while i < n and j < m:\n        d = qmz[i] - cmz[j]\n        if d < -tol:\n            buf[b] = qp[i]\n            i += 1\n        elif d > tol:\n            buf[b] = cp[j]\n            j += 1\n        else:\n            buf[b] = qp[i] + cp[j]\n            i += 1\n            j += 1\n        b += 1\n    while i < n:\n        buf[b] = qp[i]\n        i += 1\n        b += 1\n    while j < m:\n        buf[b] = cp[j]\n        j += 1\n        b += 1\n    for x in range(b):\n        tot += buf[x]\n    if tot <= 0:\n        return 0.0\n    for x in range(b):\n        v = buf[x] / tot\n        if v > 0:\n            SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search(qmz, qp, cand, off, allmz, allin, tol):\n    \"\"\"前処理済みライブラリの cand 行に対する entropy similarity。\"\"\"\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]\n        a = off[c]\n        b = off[c + 1]\n        if b > a:\n            out[k] = entropy_sim(qmz, qp, allmz[a:b], allin[a:b], tol)\n    return out\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift(qmz, qp, cand, off, allmz, allin, tol, shift):\n    \"\"\"直接一致と precursor 質量差だけ shift した一致の大きい方 (analog 伝播用)。\"\"\"\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]\n        a = off[c]\n        b = off[c + 1]\n        if b <= a:\n            continue\n        cm = allmz[a:b]\n        cp = allin[a:b]\n        s1 = entropy_sim(qmz, qp, cm, cp, tol)\n        sh = shift[k]\n        if -0.001 < sh < 0.001:\n            out[k] = s1\n            continue\n        s2 = entropy_sim(qmz, qp, cm + sh, cp, tol)\n        out[k] = s1 if s1 > s2 else s2\n    return out\n\n\n@njit(cache=True, parallel=True)\ndef _clean_counts(off, mz, it, floor, topk, power, ent_weight):\n    n = len(off) - 1\n    cnt = np.zeros(n, np.int64)\n    for r in prange(n):\n        om, _ = clean(\n            mz[off[r] : off[r + 1]], it[off[r] : off[r + 1]], floor, topk, power, ent_weight\n        )\n        cnt[r] = len(om)\n    return cnt\n\n\n@njit(cache=True, parallel=True)\ndef _clean_fill(off, mz, it, new_off, out_mz, out_it, floor, topk, power, ent_weight):\n    n = len(off) - 1\n    for r in prange(n):\n        om, oi = clean(\n            mz[off[r] : off[r + 1]], it[off[r] : off[r + 1]], floor, topk, power, ent_weight\n        )\n        a = new_off[r]\n        for i in range(len(om)):\n            out_mz[a + i] = om[i]\n            out_it[a + i] = oi[i]\n\n\ndef drop_precursor_region(\n    off: np.ndarray, mz: np.ndarray, it: np.ndarray, prec: np.ndarray, window: float\n) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    \"\"\"各スペクトルから m/z > prec - window のピークを除く (flat 配列のまま)。\"\"\"\n    row = np.repeat(np.arange(len(off) - 1), np.diff(off))\n    keep = mz <= (prec[row] - window)\n    cnt = np.bincount(row[keep], minlength=len(off) - 1)\n    new_off = np.zeros(len(off), np.int64)\n    np.cumsum(cnt, out=new_off[1:])\n    return new_off, mz[keep], it[keep]\n\n\ndef clean_flat(\n    off: np.ndarray, mz: np.ndarray, it: np.ndarray, cfg: CleanConfig\n) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    \"\"\"flat 配列 (offsets + values) のスペクトル群を一括 clean。\"\"\"\n    args = (cfg.floor, cfg.topk, cfg.power, cfg.ent_weight)\n    cnt = _clean_counts(off, mz, it, *args)\n    new_off = np.zeros(len(cnt) + 1, np.int64)\n    np.cumsum(cnt, out=new_off[1:])\n    out_mz = np.empty(new_off[-1], np.float32)\n    out_it = np.empty(new_off[-1], np.float32)\n    _clean_fill(off, mz, it, new_off, out_mz, out_it, *args)\n    return new_off, out_mz, out_it\n\n\n@dataclass\nclass CleanLibrary:\n    \"\"\"train.parquet 全行の前処理済みスペクトル。行番号は train.parquet の行順と一致。\"\"\"\n\n    off: np.ndarray\n    mz: np.ndarray\n    it: np.ndarray\n    nm: np.ndarray  # neutral mass (adduct 不明なら nan)\n\n    @property\n    def n_peaks(self) -> np.ndarray:\n        return np.diff(self.off)\n\n\ndef clean_library(train_path: Path, cache_path: Path | None, cfg: CleanConfig) -> CleanLibrary:\n    \"\"\"row group ごとに読んで clean し npz にキャッシュする (全 4 億ピークの一括読込を避ける)。\"\"\"\n    if cache_path is not None and cache_path.exists():\n        z = np.load(cache_path)\n        return CleanLibrary(off=z[\"off\"], mz=z[\"mz\"], it=z[\"it\"], nm=z[\"nm\"])\n    pf = pq.ParquetFile(train_path)\n    offs, mzs, its, nms = [np.zeros(1, np.int64)], [], [], []\n    base = 0\n    for g in range(pf.num_row_groups):\n        t = pf.read_row_group(\n            g, columns=[\"ms2_mzs\", \"ms2_normalized_intensities\", \"precursor_mz\", \"adduct\"]\n        )\n        mzc = t.column(\"ms2_mzs\").combine_chunks()\n        itc = t.column(\"ms2_normalized_intensities\").combine_chunks()\n        off = mzc.offsets.to_numpy().astype(np.int64)\n        off = off - off[0]\n        mz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)\n        it = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)\n        prec = t.column(\"precursor_mz\").to_numpy(zero_copy_only=False).astype(np.float64)\n        if cfg.precursor_window > 0:\n            off, mz, it = drop_precursor_region(off, mz, it, prec, cfg.precursor_window)\n        o2, m2, i2 = clean_flat(off, mz, it, cfg)\n        offs.append(o2[1:] + base)\n        base += int(o2[-1])\n        mzs.append(m2)\n        its.append(i2)\n        add = np.asarray(t.column(\"adduct\").to_pylist(), dtype=object)\n        nms.append(neutral_mass(prec, add))\n        del t, mzc, itc, mz, it\n    lib = CleanLibrary(\n        off=np.concatenate(offs),\n        mz=np.concatenate(mzs),\n        it=np.concatenate(its),\n        nm=np.concatenate(nms),\n    )\n    if cache_path is not None:\n        cache_path.parent.mkdir(parents=True, exist_ok=True)\n        np.savez(cache_path, off=lib.off, mz=lib.mz, it=lib.it, nm=lib.nm)\n    return lib\n", "casmi26/baseline/pipeline.py": "\"\"\"fold-safe な baseline パイプライン (haideptry V17 の Channel 1〜3)。\n\nV17 との差分 (詳細は experiments/EXP-0001/README.md):\n    - ライブラリ・analog 代表スペクトル・候補 pool の train 由来構造は、すべて\n      ホールドアウト identity group を除いた側だけから作る\n    - Channel 4 (FPNet) は使わない: 公開重みは train 全体で学習済みでリークする\n    - 学習済みランカー (rank_train.npz の HistGBM) は使わない: 同じく train 全体で\n      作られている。代わりに固定重みの線形融合 score = w_lib*lib + w_analog*ap + w_frag*frag\n    - analog の Tanimoto は 6,930-bit 複合指紋ではなく Morgan r2 2048-bit\n\"\"\"\n\nfrom __future__ import annotations\n\nimport multiprocessing as mp\nfrom collections.abc import Callable, Sequence\nfrom dataclasses import dataclass, field\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import DataStructs\n\nfrom casmi26.baseline.chem import morgan_fp\nfrom casmi26.baseline.fragments import explain_score, fragment_masses\nfrom casmi26.baseline.spectra import CleanConfig, CleanLibrary, clean, search, search_shift\nfrom casmi26.metrics.breakdown import canonical_inchikey14\n\n\n@dataclass(frozen=True)\nclass BaselineConfig:\n    ppm_win: float = 8.5  # ライブラリ検索 (Channel 1) の窓。候補窓も pool_ppm 未指定ならこれ\n    ppm_fallback: float = 30.0\n    # 候補窓: 半幅 = max(target × pool_ppm, pool_min_da)。None / 0 は V17 と同じ\n    pool_ppm: float | None = None\n    pool_min_da: float = 0.0\n    # precursor m/z の系統ずれ (Da)。中性質量を出す前に引く。0 は V17 と同じ\n    mz_offset_pos: float = 0.0\n    mz_offset_neg: float = 0.0\n    analog_win: float = 200.0\n    n_analog: int = 100\n    sim_power: float = 4.0\n    w_lib: float = 1.0\n    w_analog: float = 1.0\n    w_frag: float = 0.25\n    workers: int = 11\n    clean: CleanConfig = field(default_factory=CleanConfig)\n\n\n@dataclass(frozen=True)\nclass Query:\n    molecule_id: str\n    truth_key: str\n    spectra: Sequence[tuple[np.ndarray, np.ndarray]]  # 生の (m/z, 強度)\n    neutral_masses: np.ndarray\n    ion_mode: float  # +1 positive / -1 negative (平均)\n    precursor_mzs: np.ndarray | None = None  # spectra と同順。precursor_window > 0 で使う\n\n\nclass SpectralIndex:\n    \"\"\"許可された行だけで作るライブラリ検索 (Channel 1) と analog 代表集合 (Channel 2)。\"\"\"\n\n    def __init__(self, lib: CleanLibrary, allowed: np.ndarray, row_keys: np.ndarray) -> None:\n        self.lib = lib\n        self.row_keys = row_keys\n        ok = allowed & np.isfinite(lib.nm) & (lib.n_peaks > 0)\n        rows = np.flatnonzero(ok)\n        o = np.argsort(lib.nm[rows], kind=\"mergesort\")\n        self.rows = rows[o]\n        self.snm = lib.nm[self.rows]\n\n        # 構造ごとにピーク数最大のスペクトルを 1 本 (V17 build_rep)\n        df = pd.DataFrame({\"row\": rows, \"key\": row_keys[rows], \"np\": lib.n_peaks[rows]})\n        rep = df.sort_values([\"key\", \"np\"], ascending=[True, False]).drop_duplicates(\"key\")\n        rep_rows = rep[\"row\"].to_numpy()\n        o = np.argsort(lib.nm[rep_rows])\n        self.rep_rows = rep_rows[o]\n        self.rep_nm = lib.nm[self.rep_rows]\n        self.rep_keys = row_keys[self.rep_rows]\n        self.keys = frozenset(rep[\"key\"].tolist())\n\n    def lib_hits(\n        self, specs: Sequence[tuple[np.ndarray, np.ndarray]], target: float, cfg: BaselineConfig\n    ) -> dict[str, float]:\n        tol = target * cfg.ppm_win / 1e6\n        lo, hi = np.searchsorted(self.snm, [target - tol, target + tol])\n        cand = self.rows[lo:hi]\n        agg: dict[str, float] = {}\n        if len(cand) == 0:\n            return agg\n        for qm, qp in specs:\n            sc = search(qm, qp, cand, self.lib.off, self.lib.mz, self.lib.it, cfg.clean.mz_tol)\n            for k, s in zip(self.row_keys[cand], sc, strict=True):\n                if s > agg.get(k, -1.0):\n                    agg[k] = float(s)\n        return agg\n\n    def analogs(\n        self, specs: Sequence[tuple[np.ndarray, np.ndarray]], target: float, cfg: BaselineConfig\n    ) -> list[tuple[str, float]]:\n        lo, hi = np.searchsorted(self.rep_nm, [target - cfg.analog_win, target + cfg.analog_win])\n        cand = self.rep_rows[lo:hi]\n        if len(cand) == 0:\n            return []\n        shift = (target - self.rep_nm[lo:hi]).astype(np.float32)\n        best = np.zeros(len(cand), np.float32)\n        for qm, qp in specs:\n            sc = search_shift(\n                qm, qp, cand, self.lib.off, self.lib.mz, self.lib.it, cfg.clean.mz_tol, shift\n            )\n            np.maximum(best, sc, out=best)\n        top = np.argsort(-best)[: cfg.n_analog]\n        return [(str(self.rep_keys[lo + i]), float(best[i])) for i in top]\n\n\nclass CandidatePool:\n    \"\"\"COCONUT ∪ (ホールドアウト外の train 構造) を中性質量でソートした候補 pool。\n\n    known_key は train 由来構造の scorer_key (公式 RDKit で正規化済み)。\n    COCONUT 由来は None で、窓に入ったものだけ canonical_inchikey14 を計算する。\n    \"\"\"\n\n    def __init__(\n        self,\n        mass: np.ndarray,\n        smiles: np.ndarray,\n        known_key: np.ndarray,\n        source: np.ndarray,\n        prior: np.ndarray | None = None,\n    ) -> None:\n        o = np.argsort(mass, kind=\"mergesort\")\n        self.mass = mass[o]\n        self.smiles = smiles[o]\n        self.known_key = known_key[o]\n        self.source = source[o]\n        # 文献量の prior (PubChem 由来の候補だけ有限値、他は nan)。EXP-0009\n        self.prior = prior[o] if prior is not None else None\n\n    def window(self, target: float, ppm: float, min_da: float = 0.0) -> np.ndarray:\n        half = max(target * ppm / 1e6, min_da)\n        lo = np.searchsorted(self.mass, target - half, \"left\")\n        hi = np.searchsorted(self.mass, target + half, \"right\")\n        return np.arange(lo, hi)\n\n\ndef _z(x: np.ndarray) -> np.ndarray:\n    \"\"\"query 内で標準化 (分散 0 なら 0)。\"\"\"\n    sd = float(x.std())\n    return (x - x.mean()) / sd if sd > 1e-9 else np.zeros_like(x)\n\n\ndef _pmap(fn: Callable, items: list, workers: int, chunksize: int) -> list:\n    if workers <= 1 or len(items) < 200:\n        return [fn(x) for x in items]\n    # numba のスレッドを抱えた親から fork しないよう spawn を使う\n    with mp.get_context(\"spawn\").Pool(workers) as p:\n        return p.map(fn, items, chunksize=chunksize)\n\n\n@dataclass\nclass BaselineOutput:\n    stages: dict[str, pd.DataFrame]\n    diag: pd.DataFrame\n    database_keys: set[str]\n    # 候補ごとの生スコア (学習済みランカー用)。順位は同点を pool 順で割るので特徴量に使わない\n    features: pd.DataFrame | None = None\n\n\ndef run_baseline(\n    queries: Sequence[Query],\n    index: SpectralIndex,\n    pool: CandidatePool,\n    key_to_smiles: dict[str, str],\n    cfg: BaselineConfig,\n    log: Callable[[str], None] = print,\n) -> BaselineOutput:\n    \"\"\"全 query を処理し、誤差分解用の段出力 (long format) を返す。\"\"\"\n    c = cfg.clean\n    # Phase 1: スペクトル検索と候補窓 (numba)\n    per_q = []\n    for i, q in enumerate(queries):\n        specs = []\n        for si, (mz, it) in enumerate(q.spectra):\n            if c.precursor_window > 0 and q.precursor_mzs is not None:\n                keep = np.asarray(mz) <= q.precursor_mzs[si] - c.precursor_window\n                mz, it = np.asarray(mz)[keep], np.asarray(it)[keep]\n            qm, qp = clean(\n                np.asarray(mz, np.float32),\n                np.asarray(it, np.float32),\n                c.floor,\n                c.topk,\n                c.power,\n                c.ent_weight,\n            )\n            if len(qm):\n                specs.append((qm, qp))\n        nms = q.neutral_masses[np.isfinite(q.neutral_masses)]\n        target = float(np.median(nms)) if len(nms) else float(\"nan\")\n        lib_hits: dict[str, float] = {}\n        analogs: list[tuple[str, float]] = []\n        cand = np.zeros(0, np.int64)\n        if np.isfinite(target):\n            lib_hits = index.lib_hits(specs, target, cfg)\n            analogs = index.analogs(specs, target, cfg)\n            pool_ppm = cfg.pool_ppm if cfg.pool_ppm is not None else cfg.ppm_win\n            cand = pool.window(target, pool_ppm, cfg.pool_min_da)\n            if len(cand) == 0:\n                cand = pool.window(target, cfg.ppm_fallback)\n        per_q.append((target, lib_hits, analogs, cand))\n        if (i + 1) % 100 == 0:\n            log(f\"  spectral search {i + 1}/{len(queries)}\")\n\n    # Phase 2: COCONUT 由来候補の canonical key (並列)\n    all_cand = np.unique(np.concatenate([p[3] for p in per_q])) if per_q else np.zeros(0, int)\n    need = [int(j) for j in all_cand if pool.known_key[j] is None]\n    log(f\"  canonicalize {len(need):,} COCONUT candidates\")\n    keys = _pmap(canonical_inchikey14, [str(pool.smiles[j]) for j in need], cfg.workers, 64)\n    cand_key: dict[int, str | None] = {int(j): pool.known_key[j] for j in all_cand}\n    cand_key.update(dict(zip(need, keys, strict=True)))\n\n    # Phase 3: 断片質量 (並列、MetFrag-lite)\n    cand_smiles = sorted({str(pool.smiles[j]) for j in all_cand})\n    log(f\"  fragment masses for {len(cand_smiles):,} candidates\")\n    frags = dict(\n        zip(cand_smiles, _pmap(fragment_masses, cand_smiles, cfg.workers, 16), strict=True)\n    )\n\n    # Phase 4: 指紋 (Tanimoto 用)\n    fp_cache: dict[str, DataStructs.ExplicitBitVect | None] = {}\n\n    def fp(smi: str) -> DataStructs.ExplicitBitVect | None:\n        if smi not in fp_cache:\n            fp_cache[smi] = morgan_fp(smi)\n        return fp_cache[smi]\n\n    # Phase 5: 特徴量と融合\n    stage_names = [\"pool\", \"ranker\", \"channel_lib\", \"channel_analog\", \"channel_frag\"]\n    if pool.prior is not None:\n        stage_names += [\"ranker_prior\", \"analog_prior\", \"analog_frag_prior\", \"prior_only\"]\n    rows: dict[str, list[tuple[str, str, int, str]]] = {n: [] for n in stage_names}\n    diag = []\n    feat_rows: list[pd.DataFrame] = []\n    for q, (target, lib_hits, analogs, cand) in zip(queries, per_q, strict=True):\n        seen: set[str] = set()\n        idx, ckeys = [], []\n        for j in cand:\n            k = cand_key.get(int(j))\n            if k is None or k in seen:\n                continue\n            seen.add(k)\n            idx.append(int(j))\n            ckeys.append(k)\n        n = len(idx)\n        if n == 0:\n            diag.append((q.molecule_id, target, 0, 0.0, 0.0))\n            continue\n        lv = np.array([lib_hits.get(k, 0.0) for k in ckeys], np.float32)\n\n        afps, w = [], []\n        for k, sim in analogs:\n            f = fp(key_to_smiles.get(k, \"\"))\n            if f is not None:\n                afps.append(f)\n                w.append(max(sim, 0.0))\n        ap = np.zeros(n, np.float32)\n        if afps:\n            wp = np.asarray(w, np.float32) ** cfg.sim_power\n            for t, ci in enumerate(idx):\n                f = fp(str(pool.smiles[ci]))\n                if f is not None:\n                    tan = np.asarray(DataStructs.BulkTanimotoSimilarity(f, afps), np.float32)\n                    ap[t] = float((tan * wp).max())\n\n        peaks = []\n        for mz, it in q.spectra:\n            m2, i2 = clean(\n                np.asarray(mz, np.float32), np.asarray(it, np.float32), c.floor, c.topk, 1.0, False\n            )\n            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n        fr = np.array(\n            [\n                max(\n                    (\n                        explain_score(frags[str(pool.smiles[j])], a, b, q.ion_mode, c.mz_tol)\n                        for a, b in peaks\n                    ),\n                    default=0.0,\n                )\n                for j in idx\n            ],\n            np.float32,\n        )\n\n        score = cfg.w_lib * lv + cfg.w_analog * ap + cfg.w_frag * fr\n        # 文献量の prior: 同じ構造キーの PubChem 候補の値。窓の上位 N 件に無い構造は、\n        # 上位 N 件の最小値 (= それ以下) とみなす\n        pr = np.zeros(n, np.float32)\n        if pool.prior is not None:\n            best: dict[str, float] = {}\n            for j in cand:\n                k, v = cand_key.get(int(j)), float(pool.prior[int(j)])\n                if k is not None and np.isfinite(v):\n                    best[k] = max(best.get(k, -np.inf), v)\n            floor = min(best.values()) if best else 0.0\n            pr = np.array([best.get(k, floor) for k in ckeys], np.float32)\n        feat_rows.append(\n            pd.DataFrame(\n                {\n                    \"molecule_id\": q.molecule_id,\n                    \"candidate_key\": ckeys,\n                    \"smiles\": [str(pool.smiles[j]) for j in idx],\n                    \"source\": [str(pool.source[j]) for j in idx],\n                    \"mass_ppm\": [(float(pool.mass[j]) / target - 1) * 1e6 for j in idx],\n                    \"lib\": lv,\n                    \"analog\": ap,\n                    \"frag\": fr,\n                    \"prior\": pr,\n                }\n            )\n        )\n        ranked = [\n            (\"pool\", np.zeros(n)),\n            (\"ranker\", score),\n            (\"channel_lib\", lv),\n            (\"channel_analog\", ap),\n            (\"channel_frag\", fr),\n        ]\n        if pool.prior is not None:\n            ranked += [\n                (\"ranker_prior\", _z(score) + pr),\n                (\"analog_prior\", _z(ap) + pr),\n                (\"analog_frag_prior\", _z(ap) + 0.5 * _z(fr) + pr),\n                (\"prior_only\", pr),\n            ]\n        for name, sc in ranked:\n            order = np.argsort(-sc, kind=\"stable\")\n            rows[name].extend(\n                (q.molecule_id, ckeys[o], r, str(pool.smiles[idx[o]]))\n                for r, o in enumerate(order, start=1)\n            )\n        diag.append(\n            (q.molecule_id, target, n, float(lv.max()), float(analogs[0][1]) if analogs else 0.0)\n        )\n\n    stages = {\n        n: pd.DataFrame(v, columns=[\"molecule_id\", \"candidate_key\", \"rank\", \"smiles\"])\n        for n, v in rows.items()\n    }\n    db_keys = {k for k in cand_key.values() if k is not None}\n    return BaselineOutput(\n        stages=stages,\n        diag=pd.DataFrame(\n            diag, columns=[\"molecule_id\", \"target_mass\", \"n_candidates\", \"best_lib\", \"best_analog\"]\n        ),\n        database_keys=db_keys,\n        features=pd.concat(feat_rows, ignore_index=True) if feat_rows else None,\n    )\n", "casmi26/baseline/pubchem.py": "\"\"\"PubChem 本体 (約 1.2 億 CID) の質量順索引と、窓ごとの文献量上位 N 件の取り出し (EXP-0009)。\n\n索引は scripts/casmi26_build_pubchem_index.py が PubChem FTP の Extras から作る:\n    mass.npy     float64  質量順に並べたモノアイソトピック質量\n    cid.npy      int32    同じ順の CID\n    lsid.npy     float32  ln(1 + SID 件数)  (登録の多さ)\n    lpmid.npy    float32  ln(1 + PubMed 件数)  (文献の多さ)\n    smiles.bin   SMILES を改行区切りで CID 順に連結したもの\n    smiles_off.npy int64  CID → smiles.bin の開始位置 (長さは次の CID の位置との差)\n\nSMILES は全部メモリに載せず、窓に入った候補の分だけ memmap から読む。\n文献量の prior は公開 kernel と同じ 0.25 × (lsid + lpmid)。\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\nPRIOR_WEIGHT = 0.25\n\n\n@dataclass(frozen=True)\nclass PubChemHit:\n    cid: int\n    smiles: str\n    mass: float\n    prior: float  # PRIOR_WEIGHT × (lsid + lpmid)\n\n\nclass PubChemIndex:\n    def __init__(self, root: Path) -> None:\n        self.mass = np.load(root / \"mass.npy\", mmap_mode=\"r\")\n        self.cid = np.load(root / \"cid.npy\", mmap_mode=\"r\")\n        self.prior = PRIOR_WEIGHT * (\n            np.load(root / \"lsid.npy\", mmap_mode=\"r\").astype(np.float32)\n            + np.load(root / \"lpmid.npy\", mmap_mode=\"r\").astype(np.float32)\n        )\n        # 完全版は CID 添字の位置 (smiles_off.npy)、軽量版は質量順の並び添字 (smiles_pos_off.npy)\n        self.by_pos = (root / \"smiles_pos_off.npy\").exists()\n        self.off = np.load(\n            root / (\"smiles_pos_off.npy\" if self.by_pos else \"smiles_off.npy\"), mmap_mode=\"r\"\n        )\n        self.smi = np.memmap(root / \"smiles.bin\", dtype=np.uint8, mode=\"r\")\n\n    def __len__(self) -> int:\n        return len(self.mass)\n\n    def smiles_of(self, cid: int) -> str:\n        \"\"\"完全版のみ: CID から SMILES。\"\"\"\n        a, b = int(self.off[cid]), int(self.off[cid + 1])\n        return bytes(self.smi[a : b - 1]).decode(\"ascii\", errors=\"replace\")\n\n    def smiles_at(self, pos: int) -> str:\n        \"\"\"質量順の並び添字から SMILES (完全版・軽量版の両方)。\"\"\"\n        if not self.by_pos:\n            return self.smiles_of(int(self.cid[pos]))\n        a, b = int(self.off[pos]), int(self.off[pos + 1])\n        return bytes(self.smi[a : b - 1]).decode(\"ascii\", errors=\"replace\")\n\n    def top_n(self, target: float, half: float, n: int) -> list[PubChemHit]:\n        \"\"\"target ± half の窓から、文献量 prior の上位 n 件。\"\"\"\n        lo = int(np.searchsorted(self.mass, target - half, \"left\"))\n        hi = int(np.searchsorted(self.mass, target + half, \"right\"))\n        if hi <= lo or n <= 0:\n            return []\n        pri = np.asarray(self.prior[lo:hi])\n        k = min(n, hi - lo)\n        top = np.argpartition(-pri, k - 1)[:k] if k < hi - lo else np.arange(hi - lo)\n        top = top[np.argsort(-pri[top], kind=\"stable\")]\n        out = []\n        for t in top:\n            out.append(\n                PubChemHit(\n                    int(self.cid[lo + t]),\n                    self.smiles_at(lo + t),\n                    float(self.mass[lo + t]),\n                    float(pri[t]),\n                )\n            )\n        return out\n\n    def window_size(self, target: float, half: float) -> int:\n        lo = int(np.searchsorted(self.mass, target - half, \"left\"))\n        hi = int(np.searchsorted(self.mass, target + half, \"right\"))\n        return hi - lo\n\n\ndef collect_candidates(\n    index: PubChemIndex,\n    neutral_masses: list[np.ndarray],\n    *,\n    pool_ppm: float,\n    pool_min_da: float,\n    top: int,\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[int]]:\n    \"\"\"各 query の窓 (パイプラインと同じ中心・幅) から文献量の上位 top 件を集める。\n\n    返り値: (smiles, mass, prior, 窓の大きさのリスト)。CID で重複を除く。\n    \"\"\"\n    hits: dict[int, tuple[str, float, float]] = {}\n    sizes: list[int] = []\n    for nm in neutral_masses:\n        v = nm[np.isfinite(nm)]\n        if not len(v):\n            continue\n        t = float(np.median(v))\n        half = max(t * pool_ppm / 1e6, pool_min_da)\n        sizes.append(index.window_size(t, half))\n        for h in index.top_n(t, half, top):\n            hits[h.cid] = (h.smiles, h.mass, h.prior)\n    return (\n        np.array([v[0] for v in hits.values()], object),\n        np.array([v[1] for v in hits.values()], np.float64),\n        np.array([v[2] for v in hits.values()], np.float32),\n        sizes,\n    )\n", "casmi26/baseline/infer.py": "\"\"\"test 推論: train 全体をライブラリ・pool にして submission.csv を作る。\n\nCV (scripts/casmi26_baseline_cv.py) との違いはホールドアウトがないことだけで、\nライブラリ / analog 代表 / 候補 pool / 特徴量 / 融合はすべて同じコードを通る。\n\"\"\"\n\nfrom __future__ import annotations\n\nimport pickle\nfrom collections.abc import Callable\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\n\nfrom casmi26.baseline.chem import corrected_mz, neutral_mass\nfrom casmi26.baseline.pipeline import (\n    BaselineConfig,\n    BaselineOutput,\n    CandidatePool,\n    Query,\n    SpectralIndex,\n    run_baseline,\n)\nfrom casmi26.baseline.pubchem import PubChemIndex, collect_candidates\nfrom casmi26.baseline.spectra import clean_library\n\nTOPN = 25\nPAD_SMILES = \"CCO\"\n\n# 提出モード → 使う段出力\nMODES = {\n    \"fusion\": \"ranker\",\n    \"analog\": \"channel_analog\",\n    # EXP-0009: PubChem 候補 + 文献量 prior を使う段 (pubchem_dir 指定時のみ)\n    \"analog_frag_prior\": \"analog_frag_prior\",\n    \"prior_only\": \"prior_only\",\n}\n\n\n@dataclass(frozen=True)\nclass InputPaths:\n    train: Path\n    test: Path\n    sample_submission: Path\n    coconut_dir: Path  # coco_meta.pkl / coco_mass.npy\n    structures: Path  # casmi26-identity-disjoint-validation-folds/structures.parquet\n    pubchemlite_dir: Path | None = None  # pubchemlite_meta.pkl / pubchemlite_mass.npy (EXP-0008)\n    pubchem_dir: Path | None = None  # casmi26.baseline.pubchem の索引 (EXP-0009、軽量版で可)\n\n\n# EXP-0008 の候補 pool 設定: m/z のずれ補正 (Enveda timsTOF) + ±max(5 ppm, 2 mDa)\nPOOL_EXP0008 = dict(pool_ppm=5.0, pool_min_da=2e-3, mz_offset_pos=0.70e-3, mz_offset_neg=0.25e-3)\n\n\ndef build_queries(test: pd.DataFrame, cfg: BaselineConfig) -> list[Query]:\n    queries = []\n    for mid, sub in test.groupby(\"molecule_id\", sort=True):\n        pmz = corrected_mz(\n            sub[\"precursor_mz\"].to_numpy(np.float64),\n            sub[\"ionization_mode\"].to_numpy(),\n            cfg.mz_offset_pos,\n            cfg.mz_offset_neg,\n        )\n        nm = neutral_mass(pmz, sub[\"adduct\"].to_numpy())\n        mode = float(np.mean([1.0 if m == \"positive\" else -1.0 for m in sub[\"ionization_mode\"]]))\n        spectra = [\n            (np.asarray(m, np.float32), np.asarray(i, np.float32))\n            for m, i in zip(sub[\"ms2_mzs\"], sub[\"ms2_normalized_intensities\"], strict=True)\n        ]\n        queries.append(\n            Query(\n                molecule_id=str(mid),\n                truth_key=\"\",\n                spectra=spectra,\n                neutral_masses=nm,\n                ion_mode=mode,\n                precursor_mzs=sub[\"precursor_mz\"].to_numpy(np.float64),\n            )\n        )\n    return queries\n\n\ndef run_test(\n    paths: InputPaths,\n    cfg: BaselineConfig,\n    *,\n    cache_path: Path | None = None,\n    pubchem_top: int = 100,\n    log: Callable[[str], None] = print,\n) -> BaselineOutput:\n    st = pd.read_parquet(\n        paths.structures, columns=[\"inchikey14\", \"scorer_key\", \"smiles\", \"mono_mass\"]\n    )\n    key_of_ik = dict(zip(st[\"inchikey14\"], st[\"scorer_key\"], strict=True))\n    train_ik = pq.read_table(paths.train, columns=[\"inchikey14\"]).column(\"inchikey14\").to_pylist()\n    row_keys = np.asarray([key_of_ik.get(k) for k in train_ik], dtype=object)\n    log(f\"train: {len(row_keys):,} spectra / {len(st):,} structures\")\n\n    lib = clean_library(paths.train, cache_path, cfg.clean)\n    allowed = np.array([k is not None for k in row_keys])\n    index = SpectralIndex(lib, allowed, row_keys)\n    log(f\"library: {len(index.rows):,} spectra / analog reps: {len(index.rep_rows):,}\")\n\n    with open(paths.coconut_dir / \"coco_meta.pkl\", \"rb\") as f:\n        cm = pickle.load(f)\n    co_mass = np.load(paths.coconut_dir / \"coco_mass.npy\")\n    st_ok = st.drop_duplicates(\"scorer_key\")\n    ex_mass, ex_smiles = np.zeros(0), np.zeros(0, object)\n    if paths.pubchemlite_dir is not None:\n        with open(paths.pubchemlite_dir / \"pubchemlite_meta.pkl\", \"rb\") as f:\n            pm = pickle.load(f)\n        ex_mass = np.load(paths.pubchemlite_dir / \"pubchemlite_mass.npy\")\n        ex_smiles = np.asarray(pm[\"smiles\"], object)\n    test = pd.read_parquet(paths.test)\n    queries = build_queries(test, cfg)\n    pc_smiles, pc_mass = np.zeros(0, object), np.zeros(0)\n    pc_prior = np.zeros(0, np.float32)\n    if paths.pubchem_dir is not None:\n        pc_smiles, pc_mass, pc_prior, sizes = collect_candidates(\n            PubChemIndex(paths.pubchem_dir),\n            [q.neutral_masses for q in queries],\n            pool_ppm=cfg.pool_ppm if cfg.pool_ppm is not None else cfg.ppm_win,\n            pool_min_da=cfg.pool_min_da,\n            top=pubchem_top,\n        )\n        log(f\"PubChem: window median {np.median(sizes):.0f} → {len(pc_mass):,} candidates\")\n    n_base = len(co_mass) + len(ex_mass) + len(st_ok)\n    pool = CandidatePool(\n        mass=np.concatenate([co_mass, ex_mass, st_ok[\"mono_mass\"].to_numpy(), pc_mass]),\n        smiles=np.concatenate(\n            [\n                np.asarray(cm[\"smiles\"], object),\n                ex_smiles,\n                st_ok[\"smiles\"].to_numpy(object),\n                pc_smiles,\n            ]\n        ),\n        known_key=np.concatenate(\n            [\n                np.full(len(co_mass) + len(ex_mass), None, object),\n                st_ok[\"scorer_key\"].to_numpy(object),\n                np.full(len(pc_mass), None, object),\n            ]\n        ),\n        source=np.array(\n            [\"coconut\"] * len(co_mass)\n            + [\"pubchemlite\"] * len(ex_mass)\n            + [\"train\"] * len(st_ok)\n            + [\"pubchem\"] * len(pc_mass),\n            object,\n        ),\n        prior=(\n            np.concatenate([np.full(n_base, np.nan, np.float32), pc_prior])\n            if paths.pubchem_dir is not None\n            else None\n        ),\n    )\n    log(f\"pool: {len(co_mass):,} COCONUT + {len(ex_mass):,} PubChemLite + {len(st_ok):,} train\")\n\n    log(f\"test: {len(queries)} molecules / {len(test)} spectra\")\n    key_to_smiles = dict(zip(st[\"scorer_key\"], st[\"smiles\"], strict=False))\n    return run_baseline(queries, index, pool, key_to_smiles, cfg, log=log)\n\n\ndef to_submission(stage: pd.DataFrame, sample: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"段出力 (long format) → submission。候補が 25 本に満たない分子は PAD_SMILES で埋める。\"\"\"\n    top = stage[stage[\"rank\"] <= TOPN].sort_values([\"molecule_id\", \"rank\"])\n    lists = top.groupby(\"molecule_id\")[\"smiles\"].apply(list).to_dict()\n    rows = []\n    for mid in sample[\"molecule_id\"]:\n        smis = list(lists.get(mid, []))[:TOPN]\n        smis += [PAD_SMILES] * (TOPN - len(smis))\n        rows.append((mid, \";\".join(smis)))\n    return pd.DataFrame(rows, columns=[\"molecule_id\", \"smiles\"])\n\n\ndef validate_submission(sub: pd.DataFrame, sample: pd.DataFrame) -> None:\n    assert list(sub.columns) == [\"molecule_id\", \"smiles\"]\n    assert len(sub) == len(sample)\n    assert (sub[\"molecule_id\"].to_numpy() == sample[\"molecule_id\"].to_numpy()).all()\n    assert sub[\"smiles\"].notna().all()\n    assert (sub[\"smiles\"].str.split(\";\").str.len() == TOPN).all()\n"}

In [ ]:
import glob, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
ROOTS = [r for r in ("/kaggle/input", os.environ.get("CASMI_INPUT")) if r and os.path.isdir(r)]
WORK = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path("data/working/kernel")
WORK.mkdir(parents=True, exist_ok=True)


def find(name, must=""):
    hits = [h for r in ROOTS for h in glob.glob(f"{r}/**/{name}", recursive=True) if must in h]
    if not hits:
        raise FileNotFoundError(f"{name} ({must}) not found under {ROOTS}")
    return Path(sorted(hits, key=len)[0])


# 公式スコアラと同じ RDKit 2026.03.3 を入れる (Kaggle 上のみ)
if os.path.isdir("/kaggle/input"):
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    whl = glob.glob("/kaggle/input/**/rdkit-2026.3.3-*.whl", recursive=True)
    whl = [w for w in whl if tag in w]
    if whl:
        pip = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"]
        subprocess.run(pip + [whl[0]], check=False)
import rdkit
print("rdkit", rdkit.__version__, "python", sys.version.split()[0])

# 埋め込んだ casmi26 パッケージを展開
PKG = WORK / "casmi26_src"
for rel, text in EMBEDDED.items():
    p = PKG / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
sys.path.insert(0, str(PKG))

In [ ]:
import pandas as pd
from casmi26.baseline.infer import (
    MODES,
    POOL_EXP0008,
    InputPaths,
    run_test,
    to_submission,
    validate_submission,
)
from casmi26.baseline.pipeline import BaselineConfig

paths = InputPaths(
    train=find("train.parquet", "enveda"),
    test=find("test.parquet", "enveda"),
    sample_submission=find("sample_submission.csv", "enveda"),
    coconut_dir=find("coco_meta.pkl").parent,
    structures=find("structures.parquet", "identity-disjoint-validation-folds"),
    pubchemlite_dir=find("pubchemlite_meta.pkl").parent if POOL != "v17" else None,
    pubchem_dir=find("smiles_pos_off.npy").parent if POOL == "exp0009" else None,
)
print(paths)
pool_cfg = POOL_EXP0008 if POOL != "v17" else {}
cfg = BaselineConfig(workers=max(1, (os.cpu_count() or 2)), **pool_cfg)
print("POOL", POOL, pool_cfg)
log = lambda m: print(f"[{time.time() - T0:7.1f}s] {m}", flush=True)
out = run_test(paths, cfg, pubchem_top=100, log=log)

sample = pd.read_csv(paths.sample_submission)
for mode, stage in MODES.items():
    if stage in out.stages:
        out.stages[stage].to_parquet(WORK / f"stage_{stage}.parquet", index=False)
sub = to_submission(out.stages[MODES[MODE]], sample)
validate_submission(sub, sample)
sub.to_csv(WORK / "submission.csv", index=False)
out.diag.to_csv(WORK / "diag.csv", index=False)
n_real = out.stages[MODES[MODE]].groupby("molecule_id").size()
n_pad = int((n_real.reindex(sample["molecule_id"]).fillna(0) < 25).sum())
log(f"MODE={MODE} submission.csv {sub.shape} / molecules padded to 25: {n_pad}")
print(out.diag["n_candidates"].describe())